# 위성영상 × AI 핸즈온: 2025 의성 산불 피해 분석
### 공주대 산림과학과 · 환경공간정보 활용 청년인재 역량 강화교육 · 2026.10.08 · Google Colab
경희대학교 김은빈 (ebkim@khu.ac.kr)

- **사례**: 2025년 3월 22일 경북 의성군 안평면에서 시작해 안동·청송·영양·영덕으로 번진 산불
- **위성**: Copernicus Sentinel-2 (ESA, 무료)
- **화점 자료**: NASA FIRMS
- **플랫폼**: Google Earth Engine + PyTorch

| 단계 | 내용 | 시간 |
|---|---|---|
| 1–2 | 환경 설정, Earth Engine 인증 | 10분 |
| 3–4 | 연구 지역 설정, 발화 전·후 영상 불러오기 | 15분 |
| 5 | NDVI·NBR·dNBR 계산, 피해 등급 지도와 면적 | 20분 |
| 6 | AI: 발화 후 영상 한 장으로 피해 찾기 (dNBR로 정답 만들기 → 기준선·MLP·U-Net 학습과 검증) | 30분 |
| 7 | 검증: FIRMS 화점과 비교, 잔설에 속는 dNBR과 속지 않는 AI | 20분 |
| | 여유·질의 | 15분 |

> **런타임**: 런타임 → 런타임 유형 변경 → **T4 GPU** 권장 (GPU가 없으면 U-Net 에폭이 자동으로 2로 줄어든다)

## 학습 목표
1. Google Earth Engine으로 Sentinel-2 위성영상을 찾아 산불 전·후 영상을 만든다.
2. NBR과 dNBR로 산불 피해 지역을 찾고 피해 면적을 계산한다.
3. dNBR로 만든 정답으로 AI(MLP·U-Net)를 학습시켜, 발화 후 영상 한 장만으로 피해 지역을 찾는다.
4. AI 정확도가 무엇을 뜻하는지 이해하고, 위성 화점 자료로 결과를 다시 확인한다.

> 넓은 피해지 계산은 Earth Engine 서버가 하고, AI 학습에 쓸 작은 영역만 Colab으로 받아 온다.

## 오늘의 핵심 지수: NBR과 dNBR
- **NBR (Normalized Burn Ratio, 정규탄화지수)** = (NIR − SWIR2) / (NIR + SWIR2). Sentinel-2에서는 (B8 − B12) / (B8 + B12).
  - 건강한 숲은 근적외(NIR)를 강하게, 단파적외(SWIR2)를 약하게 반사해 NBR이 크다.
  - 불에 타면 잎이 사라져 근적외가 떨어지고, 숯과 드러난 땅 때문에 단파적외가 올라가 NBR이 작아진다(음수까지).
  - 강의 자료의 LA 산불 사례: 탄 곳의 NBR이 +0.33에서 −0.33으로 떨어졌다.
- **dNBR (differenced NBR, 탄화지수 변화량)** = 발화 전 NBR − 발화 후 NBR.
  - 같은 자리의 NBR이 얼마나 떨어졌는지 본다. 0 근처면 변화 없음, 클수록 피해가 크다.
  - 이 실습에서는 0.27을 넘으면 '중간 이상 피해'로 본다(섹션 5의 등급표).

## 핵심 개념
| 용어 | 설명 |
|---|---|
| **Sentinel-2** | ESA 운영, 10~60 m 해상도, 13개 밴드 광학 위성, 약 5일 주기 |
| **NDVI** | Normalized Difference Vegetation Index(정규식생지수). (NIR−Red)/(NIR+Red). 잎의 양과 활력 (−1~1) |
| **NBR** | Normalized Burn Ratio(정규탄화지수). (NIR−SWIR2)/(NIR+SWIR2). 탄 흔적에 민감 |
| **dNBR** | differenced NBR(탄화지수 변화량). 발화 전 NBR − 발화 후 NBR. 클수록 피해가 크다 |
| **지수 자동 레이블** (pseudo-label) | 현장 조사 없이 지수(dNBR)로 자동으로 만든 정답. 편하지만 지수의 오류도 함께 배운다 |
| **FIRMS** | NASA 화점 자료. 위성이 감지한 열 이상 지점. dNBR과 **독립된** 확인 자료 |

## Sentinel-2 주요 밴드
| 밴드 | 이름 | 파장(nm) | 해상도(m) | 쓰임 |
|---|---|---|---|---|
| B2 | Blue | 490 | 10 | 수계, 대기 |
| B3 | Green | 560 | 10 | 식생 활력, 눈 판별 |
| B4 | Red | 665 | 10 | 엽록소 흡수 |
| B8 | NIR | 842 | 10 | 식생 생체량 |
| B11 | SWIR1 | 1610 | 20 | 수분, 눈 판별 |
| B12 | SWIR2 | 2190 | 20 | **연소 흔적 핵심** |

---
## 섹션 1. 환경 설정

**코드 설명**
- `!`로 시작하는 줄은 파이썬 코드가 아니라 Colab 컴퓨터(리눅스)에 내리는 명령이다.
- `pip install`: Colab에 기본으로 없는 패키지 4개를 설치한다. `geemap`(Earth Engine 지도), `geedim`(Earth Engine 영상 내려받기 보조), `rasterio`(GeoTIFF 파일 읽기), `segmentation-models-pytorch`(U-Net 모델).
- `apt-get install fonts-nanum`: 나눔 글꼴을 설치한다. 그래프에 한글을 쓰기 위해서다.
- 1분 안팎 걸린다. `설치 완료`가 나오면 다음 셀로 간다.

In [ ]:
# 패키지 설치 (Colab에 없는 것만) + 한글 폰트
!pip install -q geemap geedim rasterio segmentation-models-pytorch
!apt-get -qq install fonts-nanum > /dev/null
print("설치 완료")

**코드 설명**
- 실습에 쓰는 라이브러리를 불러온다. `numpy`(배열 계산), `matplotlib`(그래프), `rasterio`(GeoTIFF), `torch`(딥러닝, PyTorch).
- 설치한 나눔 글꼴을 그래프 기본 글꼴로 등록한다. 이게 없으면 그래프의 한글이 네모(□)로 깨진다.
- `device`: GPU가 잡혀 있으면 `cuda`, 없으면 `cpu`로 계산한다. 출력에 `장치: cuda`가 나오는지 확인하자. `cpu`라면 런타임 유형을 T4 GPU로 바꾼다.

In [ ]:
import os, io, math, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import font_manager
from matplotlib.colors import BoundaryNorm, ListedColormap
import requests
from PIL import Image as PILImage
import rasterio

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

# 한글 폰트 등록 (없으면 그래프 한글이 네모로 깨진다)
for path in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
             "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf"]:
    if os.path.exists(path):
        font_manager.fontManager.addfont(path)
        matplotlib.rc("font", family=font_manager.FontProperties(fname=path).get_name())
        break
matplotlib.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | 장치: {device}")

---
## 섹션 2. Google Earth Engine 인증

**미리 해 올 것**
1. https://earthengine.google.com 에서 **비영리(Noncommercial)** 용도로 등록
2. Google Cloud 프로젝트를 하나 만들고 Earth Engine API를 켠다 → **프로젝트 ID** 확인
3. 아래 `EE_PROJECT`에 프로젝트 ID를 넣고 실행 → 뜨는 창에서 구글 로그인·허용

**코드 설명**
- `EE_PROJECT = ""`의 따옴표 안에 본인 Cloud 프로젝트 ID를 넣는다. 비워 두면 `assert`가 실행을 멈추고 안내 문구를 띄운다.
- `ee.Authenticate()`: 구글 계정으로 로그인해 Earth Engine 사용 권한을 받는다. 팝업 창에서 계정을 고르고 **허용**을 누른다.
- `ee.Initialize(project=...)`: 이 프로젝트 이름으로 Earth Engine 서버에 연결한다.
- `ee.Number(42).getInfo()`: 서버에 숫자 42를 보내고 돌려받는 연결 시험이다. `연결 확인: 42`가 나오면 성공이다.

In [ ]:
# [GEE]
import ee
import geemap

EE_PROJECT = ""   # ← 본인 Cloud 프로젝트 ID (예: "my-ee-project-123")
assert EE_PROJECT, "EE_PROJECT에 본인 프로젝트 ID를 넣어 주세요."

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print("연결 확인:", ee.Number(42).getInfo())

---
## 섹션 3. 연구 지역: 2025 의성 산불

| 항목 | 내용 |
|---|---|
| **발화** | 2025년 3월 22일, 경북 의성군 안평면 |
| **확산** | 강풍을 타고 안동·청송·영양·영덕으로 확산, 3월 28일 주불 진화 |
| **피해면적** | 집계 시점·기준마다 다르다. 중앙재난안전대책본부 3월 27일 기준 경북 45,157 ha(잠정) → 산림청 정밀조사(2025년 4월 발표, 잠정) 의성·안동·청송·영양·영덕 99,289 ha (강의 자료의 값) |

**영상 고르기**: 구름이 거의 없고 계절이 같은 두 날을 골랐다.
- 발화 전: **2025-03-14** (타일별 운량 0~1%)
- 발화 후: **2025-04-08** (타일별 운량 0~4%)

**영역 세 종류** (아래 지도의 색과 같다)
- `ROI` (Region Of Interest, 관심 영역, **파란 상자**): 피해지 전체. 섹션 5에서 지수와 피해 면적을 계산한다.
- `TRAIN_BOXES` (학습 상자, **노란 상자 4개**, 지도에 A~D로 표시): **AI 학습용.** AI는 이 상자 속 픽셀로만 배운다.
- `ROI_DL` (ROI for Deep Learning, **빨간 상자**): **AI 검증용.** 학습 때는 보여주지 않고, 다 배운 AI를 여기서 채점하고 결과 지도를 그린다 (의성 북부~안동 남부, 약 22 km × 18 km).

**코드 설명**
- `ee.Geometry.Rectangle([서쪽 경도, 남쪽 위도, 동쪽 경도, 북쪽 위도])`: 경위도로 사각형 영역을 만든다.
- 영역 세 종류(`ROI`, `ROI_DL`, `TRAIN_BOXES`)를 만든다. 학습 상자 옆 괄호의 비율은 dNBR이 0.27을 넘는 픽셀의 비율로, 미리 계산해 둔 값이다.
- 날짜: `filterDate`는 끝 날짜를 포함하지 않는다. 하루만 고르려고 끝 날짜를 다음 날로 둔다.
- `BANDS`: 이번 실습에 쓰는 밴드 6개(파랑·초록·빨강·근적외·단파적외 1·2).
- `outline()`: 상자를 속이 빈 테두리로 그리는 작은 함수다.
- `label()`: 상자 가운데에 글자(학습 상자 A~D, 검증)를 찍는다. 지도 라이브러리(ipyleaflet)의 글자 표식(`DivIcon`)을 쓴다.
- `geemap.Map`: 확대·이동이 되는 지도. 배경은 구글 위성사진(`HYBRID`)이다. 오른쪽 위 레이어 버튼으로 상자를 켜고 끌 수 있다. 지도가 안 보여도 뒤 계산에는 영향이 없다.

In [ ]:
# [GEE]
ROI    = ee.Geometry.Rectangle([128.40, 36.25, 129.50, 36.80])   # 피해지 전체
ROI_DL_BOX = [128.60, 36.42, 128.85, 36.58]
ROI_DL = ee.Geometry.Rectangle(ROI_DL_BOX)                         # AI 검증 영역 (빨강)

# AI 학습 상자 4개 (괄호는 dNBR>0.27 비율, 90 m로 미리 계산)
TRAIN_BOXES = {"A": [128.85, 36.41, 129.10, 36.57],   # 검증 영역 동쪽 (피해 약 40%)
               "B": [129.10, 36.41, 129.35, 36.57],   # 더 동쪽 (약 23%)
               "C": [128.60, 36.25, 128.85, 36.41],   # 검증 영역 남쪽 (약 10%)
               "D": [128.85, 36.25, 129.10, 36.41]}   # 남동쪽 (약 5%)

PRE_START,  PRE_END  = "2025-03-14", "2025-03-15"   # 끝 날짜는 포함 안 됨
POST_START, POST_END = "2025-04-08", "2025-04-09"

BANDS  = ["B2", "B3", "B4", "B8", "B11", "B12"]
CENTER = [36.50, 128.95]

def outline(geom, color):   # 속은 비우고 테두리만 그린다
    return ee.FeatureCollection([ee.Feature(geom)]).style(color=color, fillColor="00000000", width=2)

from ipyleaflet import Marker, DivIcon
def label(box, text, color):   # 상자 가운데에 글자를 찍는다
    lon, lat = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
    html = (f'<div style="font-size:20px;font-weight:bold;color:{color};white-space:nowrap;'
            f'text-shadow:0 0 3px #000,0 0 3px #000">{text}</div>')
    return Marker(location=(lat, lon), icon=DivIcon(html=html, icon_size=(60, 30), icon_anchor=(30, 15)), draggable=False)

# 기본 배경지도(OpenStreetMap)는 Colab에서 막힌다(403). 구글 위성 배경을 쓴다
Map = geemap.Map(center=CENTER, zoom=9, basemap="HYBRID")
Map.add_layer(outline(ROI,    "0000FF"), {}, "전체 분석 영역")
Map.add_layer(outline(ROI_DL, "FF0000"), {}, "AI 검증 영역")
Map.add(label(ROI_DL_BOX, "검증", "#FF4040"))
for name, box in TRAIN_BOXES.items():
    Map.add_layer(outline(ee.Geometry.Rectangle(box), "FFFF00"), {}, f"AI 학습 상자 {name}")
    Map.add(label(box, name, "#FFFF00"))
Map

---
## 섹션 4. Sentinel-2 영상 불러오기

**이 섹션에서 만드는 것**: 발화 전(3/14) 영상 한 장과 발화 후(4/8) 영상 한 장. 둘 다 분석 영역(`ROI`) 크기이고, 구름과 물은 지운 상태다.

### 불러오는 데이터
| 항목 | 내용 |
|---|---|
| 컬렉션 | `COPERNICUS/S2_SR_HARMONIZED`: Sentinel-2 Level-2A. 대기 보정을 마친 **지표 반사율** |
| 값 | 반사율 × 10000 (예: 1600 = 반사율 0.16) |
| 쓰는 밴드 | B2·B3·B4(파랑·초록·빨강), B8(근적외), B11·B12(단파적외 1·2). 맨 위 'Sentinel-2 주요 밴드' 표 참고 |
| 함께 쓰는 밴드 | `SCL`(장면 분류 지도, 아래 설명) |
| 날짜 | 발화 전 2025-03-14 (장면 3개), 발화 후 2025-04-08 (장면 2개) |
| 'HARMONIZED'의 뜻 | 2022년 1월 ESA 처리 방식이 바뀌며 생긴 값 차이(+1000)를 맞춰 둔 판. 날짜가 달라도 값을 그대로 비교할 수 있다 |

### SCL이란?
`SCL`(Scene Classification Layer, 장면 분류 지도)은 ESA가 대기 보정을 하면서 함께 만든 **픽셀별 분류표**다.
영상마다 붙어 있고(20 m), 픽셀마다 '이 픽셀은 무엇인가'를 숫자 하나로 적어 둔다.

| 값 | 뜻 | 이 실습에서 |
|---|---|---|
| 1 | 포화·결함 픽셀 | **지운다** |
| 2 | 어두운 픽셀(지형 그림자 등) | 둔다 |
| 3 | 구름 그림자 | **지운다** |
| 4 | 식생 | 둔다 |
| 5 | 식생 없는 땅 | 둔다 |
| 6 | 물 | **지운다** (발화 전 영상 기준) |
| 7 | 분류 안 됨 | 둔다 |
| 8 · 9 | 구름 (중간 · 높은 확률) | **지운다** |
| 10 | 얇은 권운 | **지운다** |
| 11 | 눈·얼음 | **일부러 둔다** (섹션 7에서 눈이 만드는 오류를 본다) |

- 물을 지우는 이유: 분석 영역 동쪽 끝은 바다(영덕 앞바다)이고 안에 호수·강도 있다. 물은 산불과 관계없고 날마다 반사가 달라 dNBR에 잡음을 만든다.
- 물은 **발화 전** `SCL`로만 판정한다. SCL도 자동 분류라 틀릴 때가 있다. 발화 후 영상에서는 어둡게 탄 땅을 물로 잘못 분류해서, 그대로 쓰면 피해지가 지워진다.

### 장면이 여러 개인 이유와 '합치기'
- Sentinel-2 자료는 가로세로 약 100 km의 **타일** 단위로 잘려 배포된다. 분석 영역은 타일 두 개(서쪽 52SDF, 동쪽 52SEF)에 걸친다.
- 이웃 타일은 가장자리가 서로 겹치게 잘려 있다. 그래서 두 타일 경계의 띠(분석 영역의 약 10%)는 두 장면에 모두 들어 있다.
- 3/14에는 서쪽 타일을 **궤도가 다른 두 촬영**이 각각 찍었다. 궤도 103은 Sentinel-2B가 11:17에, 궤도 3은 Sentinel-2A가 11:27에 찍었다(한국 시간, 10분 차이). 그래서 3/14에는 장면이 3개이고, 분석 영역의 약 39%가 두 장면에 들어 있다.
- Sentinel-2는 같은 기종 위성 여러 대(2A·2B·2C)가 번갈아 찍는다. 4/8은 Sentinel-2C가 찍었다.
- **합치기**: 픽셀마다 찍힌 장면들의 값을 모아 **중앙값**을 쓴다. 한 장면만 있으면 그 값 그대로, 두 장면이 있으면 두 값의 평균이 된다. 이렇게 이음매 없는 영상 한 장을 만든다.

### 할 일 (코드 셀 5개)
1. **촬영 지도 보기** (`scene_list`): 장면마다 찍힌 범위(테두리)와 위성·궤도·촬영 시각을 넓은 지도에 그린다.
2. **영상 만들기** (`load_s2`, `water`): 장면 고르기 → 장면마다 구름 지우기 → 장면들을 픽셀마다 **중앙값**으로 합쳐 한 장으로 → 분석 영역으로 자르기 → 물 지우기.
3. **합치기를 눈으로 보기**: 3/14 장면 3개를 따로 그리고, 합친 결과와 '픽셀마다 찍힌 장면 수' 지도를 함께 본다.
4. **대화형 지도로 보기** (`Map2`): 발화 전·후 진색과 발화 후 SWIR 합성을 레이어로 겹쳐 본다.
5. **고정 그림으로 비교하기** (`thumb`): 발화 전·후 진색을 나란히 그리고 파일로 저장한다.

**코드 설명**
- `scene_list()`: 날짜마다 장면을 골라, 장면마다 타일 이름·궤도 번호·위성 이름(`SPACECRAFT_NAME`)·촬영 시각·찍힌 범위(`geometry()`, 장면 테두리)를 받는다.
- 두 타일이 다 들어가는 넓은 범위(`WIDE`)에 그날 장면들을 진색으로 깔고, 장면마다 테두리를 위성 색으로 그린다. 같은 날 두 번째로 지나간 궤도는 점선이다. 파란 상자가 분석 영역(`ROI`)이다.
- `thumb_ll()`: 경위도 격자 그대로 그림을 받는다. 그래야 장면 테두리(경위도 좌표)를 그림 위에 정확히 겹칠 수 있다.
- 촬영 시각은 한국 시간(UTC+9)으로 바꿔 적는다.
- 출력 읽기: 3/14에는 Sentinel-2B(궤도 103)가 두 타일을 먼저 찍고, 10분 뒤 Sentinel-2A(궤도 3)가 서쪽을 한 번 더 찍었다. 점선은 궤도 3이 찍은 범위의 동쪽 끝이다. 이 선 서쪽이 두 장면이 겹치는 곳이다. 4/8은 Sentinel-2C 한 대가 찍었다. 두 타일 사이의 세로 띠는 이웃 타일이 겹치는 부분이다.

In [ ]:
# [GEE]
# 장면마다 찍힌 범위(테두리)와 위성·궤도·촬영 시각을 넓은 지도에 그린다
import datetime as dt
from matplotlib.patches import Polygon

WIDE = [127.7, 35.95, 130.4, 37.15]                      # 두 타일이 다 들어가는 넓은 범위
WIDE_G = ee.Geometry.Rectangle(WIDE, "EPSG:4326", False)
vis_rgb = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 1600, "gamma": 1.1}   # 진색 표시 규칙
SAT_COLORS = {"Sentinel-2A": "#e41a1c", "Sentinel-2B": "#ff9900", "Sentinel-2C": "#00c8ff"}
KST = dt.timezone(dt.timedelta(hours=9))

def thumb_ll(image, region, size=1000):   # 경위도 격자 그대로 그림을 받는다
    url = image.getThumbURL({"region": region, "dimensions": size, "format": "png", "crs": "EPSG:4326"})
    return np.array(PILImage.open(io.BytesIO(requests.get(url, timeout=120).content)))

def scene_list(start, end):   # 장면마다 타일·궤도·위성·촬영 시각·찍힌 범위
    col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(ROI).filterDate(start, end)
    keys = ["MGRS_TILE", "SENSING_ORBIT_NUMBER", "SPACECRAFT_NAME", "system:time_start"]
    info = col.toList(10).map(lambda im: ee.Image(im).toDictionary(keys).set("fp", ee.Image(im).geometry())).getInfo()
    return col, sorted(info, key=lambda d: (d["system:time_start"], d["MGRS_TILE"]))

roi_xy = np.array(ROI.coordinates().getInfo()[0])
(rx0, ry0), (rx1, ry1) = roi_xy.min(0), roi_xy.max(0)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, (start, end, name) in zip(axes, [(PRE_START, PRE_END, "발화 전 3/14"), (POST_START, POST_END, "발화 후 4/8")]):
    col, info = scene_list(start, end)
    bg_wide = ee.Image(1).visualize(palette=["dddddd"]).clip(WIDE_G)
    ax.imshow(thumb_ll(bg_wide.blend(col.mosaic().visualize(**vis_rgb)), WIDE_G),
              extent=[WIDE[0], WIDE[2], WIDE[1], WIDE[3]], aspect=1 / np.cos(np.radians(36.55)))
    orbits = list(dict.fromkeys(d["SENSING_ORBIT_NUMBER"] for d in info))   # 지나간 순서대로
    for d in info:
        ring = d["fp"]["coordinates"][0]
        j = orbits.index(d["SENSING_ORBIT_NUMBER"])
        color = SAT_COLORS.get(d["SPACECRAFT_NAME"], "white")
        ax.add_patch(Polygon(ring, closed=True, fill=False, ec=color, lw=2.5, ls="-" if j == 0 else "--"))
        when = dt.datetime.fromtimestamp(d["system:time_start"] / 1000, KST)
        xs, ys = [p[0] for p in ring], [p[1] for p in ring]
        ax.text((min(xs) + max(xs)) / 2, max(ys) - 0.08 - 0.13 * j,
                f'{d["MGRS_TILE"]} · 궤도 {d["SENSING_ORBIT_NUMBER"]}\n{d["SPACECRAFT_NAME"]} · {when:%H:%M}',
                color="white", fontsize=10.5, ha="center", va="top",
                bbox=dict(fc=color, ec="none", alpha=0.85, boxstyle="round,pad=0.3"))
    ax.add_patch(mpatches.Rectangle((rx0, ry0), rx1 - rx0, ry1 - ry0, fill=False, ec="#1f4fff", lw=2.5))
    ax.text(rx0 + 0.02, ry0 + 0.02, "분석 영역 ROI", color="white", fontsize=11, va="bottom", bbox=dict(fc="#1f4fff", ec="none", pad=2))
    ax.set_xlim(WIDE[0], WIDE[2]); ax.set_ylim(WIDE[1], WIDE[3])
    ax.set_title(f"{name}: 장면 {len(info)}개 (시각은 한국 시간)", fontsize=13)
    ax.set_xlabel("경도"); ax.set_ylabel("위도")
plt.tight_layout(); plt.savefig("01_scene_map.png", dpi=110, bbox_inches="tight"); plt.show()

**코드 설명**
- `mask_clouds()`: `SCL` 밴드 값이 1(포화)·3(구름 그림자)·8·9(구름)·10(얇은 권운)인 픽셀을 지운다(`updateMask`). 눈(11)은 남긴다.
- `load_s2()`는 다섯 단계로 움직인다.
  1. 컬렉션에서 영역(`filterBounds`)과 날짜(`filterDate`)로 장면을 고른다.
  2. 장면마다 타일 이름·궤도 번호·구름 비율을 출력한다.
  3. 장면마다 구름을 가린다(`map`).
  4. 장면들을 픽셀마다 중앙값으로 합친다(`median`). 한 장면만 덮는 픽셀은 그 값 그대로, 두 장면이 겹치는 픽셀은 두 값의 평균이 된다.
  5. 분석 영역으로 자른다(`clip`).
- Earth Engine은 이 셀에서 영상을 실제로 계산하지 않는다. '계산 방법'만 기억해 두고, 지도·그림·숫자가 필요할 때 구글 서버에서 계산한다. 그래서 몇 초 만에 끝난다.
- 출력 읽기: 3/14에는 장면이 3개다. 같은 타일 52SDF가 두 궤도(103, 3)에서 찍혔다. 나란히 지나간 두 궤도의 촬영 범위가 겹친 것이다.
- `water`: 발화 전 장면들의 `SCL`에서 가장 많이 나온 값(`mode`)이 6(물)인 픽셀. `updateMask(water.Not())`로 두 영상에서 지운다. 이후 지수·면적·AI 모두 물을 빼고 계산된다. 마지막 줄에 가린 면적이 나온다.

In [ ]:
# [GEE]
CLOUD_CLASSES = [1, 3, 8, 9, 10]   # 포화, 구름그림자, 구름(중·고), 권운

def mask_clouds(img):
    scl = img.select("SCL")
    bad = scl.eq(CLOUD_CLASSES[0])
    for c in CLOUD_CLASSES[1:]:
        bad = bad.Or(scl.eq(c))
    return img.updateMask(bad.Not())

def load_s2(start, end, roi):
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterBounds(roi).filterDate(start, end))
    # 장면마다 타일 이름·궤도 번호·구름 비율을 한 번에 받아 출력한다
    scenes = col.reduceColumns(ee.Reducer.toList(3), ["MGRS_TILE", "SENSING_ORBIT_NUMBER", "CLOUDY_PIXEL_PERCENTAGE"]).get("list").getInfo()
    print(f"  {start}: 장면 {len(scenes)}개")
    for tile, orbit, cloud in scenes:
        print(f"    타일 {tile} · 궤도 {orbit} · 구름 {cloud:.1f}%")
    return col.map(mask_clouds).median().clip(roi)

print("[발화 전]"); pre_img  = load_s2(PRE_START,  PRE_END,  ROI)
print("[발화 후]"); post_img = load_s2(POST_START, POST_END, ROI)

# 물 가리기: 발화 전 SCL에서 물(6)인 곳을 두 영상 모두에서 지운다
water = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(ROI)
         .filterDate(PRE_START, PRE_END).select("SCL").mode().eq(6))
pre_img, post_img = pre_img.updateMask(water.Not()), post_img.updateMask(water.Not())
water_ha = ee.Image.pixelArea().divide(1e4).updateMask(water).reduceRegion(ee.Reducer.sum(), ROI, 30, maxPixels=1e10)
print(f"물로 가린 면적: {ee.Number(water_ha.values().get(0)).getInfo():,.0f} ha")

**코드 설명**
- `thumb()`: Earth Engine 서버에 영상을 PNG 그림으로 만들어 달라고 요청해 받아 온다(`getThumbURL`). 고정된 그림이라 나란히 비교하기 좋다. 아래 섹션에서도 계속 쓴다.
- 3/14 장면을 하나씩 꺼내(`toList`) 분석 영역 전체 틀(`on_roi`) 위에 그린다. 회색 부분은 그 장면이 찍지 않은 곳이다.
- `count()`: 픽셀마다 값이 있는 장면 수를 센다. 1이면 한 장면, 2면 두 장면이 겹친 곳이다.
- 출력 읽기: 날짜마다 '한 장면 / 두 장면' 픽셀 비율이 나온다. '합친 결과' 그림에는 이음매가 없어야 한다. 물은 앞 셀에서 지웠으므로 회색으로 보인다.

In [ ]:
# [GEE]
vis_rgb = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 1600, "gamma": 1.1}   # 진색 표시 규칙

def thumb(image, vis, region=ROI, size=900):
    url = image.getThumbURL({**vis, "region": region, "dimensions": size, "format": "png"})
    return np.array(PILImage.open(io.BytesIO(requests.get(url, timeout=120).content)))

bg = ee.Image(1).visualize(palette=["dddddd"]).clip(ROI)   # 분석 영역 전체 바탕 (회색)
def on_roi(image):               # 분석 영역 틀 위에 영상을 얹는다 (장면마다 위치를 비교하려고)
    return bg.blend(image.visualize(**vis_rgb))

def scene_count(start, end):     # 픽셀마다 찍힌 장면 수
    col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(ROI).filterDate(start, end)
    cnt = col.select("B4").count().clip(ROI)
    h = cnt.rename("n").reduceRegion(ee.Reducer.frequencyHistogram(), ROI, 60, maxPixels=1e10).get("n").getInfo()
    tot = sum(h.values())
    print(f"  {start}: " + " / ".join(f"{int(float(k))}장면 {v / tot * 100:.0f}%" for k, v in sorted(h.items())))
    return col, cnt

print("픽셀마다 찍힌 장면 수")
pre_col, pre_cnt = scene_count(PRE_START, PRE_END)
_, post_cnt      = scene_count(POST_START, POST_END)

info = pre_col.reduceColumns(ee.Reducer.toList(2), ["MGRS_TILE", "SENSING_ORBIT_NUMBER"]).get("list").getInfo()
scenes = pre_col.toList(len(info))
fig, axes = plt.subplots(2, 3, figsize=(18, 7.5))
for k, (tile, orbit) in enumerate(info):          # 3/14 장면을 하나씩
    ax = axes.flat[k]
    ax.imshow(thumb(on_roi(ee.Image(scenes.get(k))), {}))
    ax.set_title(f"3/14 장면 {k + 1}: 타일 {tile} · 궤도 {orbit}", fontsize=12)
axes.flat[3].imshow(thumb(on_roi(pre_img), {})); axes.flat[3].set_title("3/14 합친 결과 (픽셀마다 중앙값)", fontsize=12)
cmap_n, norm_n = ListedColormap(["#c6dbef", "#08519c"]), BoundaryNorm([0.5, 1.5, 2.5], 2)
for ax, cnt, t in [(axes.flat[4], pre_cnt, "3/14 찍힌 장면 수"), (axes.flat[5], post_cnt, "4/8 찍힌 장면 수")]:
    ax.imshow(thumb(cnt, {"min": 1, "max": 2, "palette": ["c6dbef", "08519c"]}))
    ax.set_title(t, fontsize=12)
    cb = plt.colorbar(plt.cm.ScalarMappable(cmap=cmap_n, norm=norm_n), ax=ax, ticks=[1, 2], fraction=0.035, pad=0.02)
    cb.ax.set_yticklabels(["1장면", "2장면"])
for a in axes.flat: a.axis("off")
plt.tight_layout(); plt.savefig("01_scenes.png", dpi=110, bbox_inches="tight"); plt.show()

**코드 설명**
- `vis_rgb`(진색): 빨강·초록·파랑 자리에 B4·B3·B2를 넣는다. 사람 눈에 보이는 색과 비슷하다. 값 0~1600(반사율 0~0.16)을 밝기 0~255로 늘린다.
- `vis_swir`(SWIR 합성): 빨강 자리에 B12(단파적외), 초록 자리에 B8(근적외), 파랑 자리에 B4를 넣는다. 건강한 숲은 근적외가 강해 초록, 탄 곳은 단파적외가 강해 갈색·붉은색으로 보인다.
- 레이어 3개를 겹쳐 올린다. 마지막에 넣은 SWIR 합성이 맨 위에 보인다. 오른쪽 위 레이어 버튼으로 하나씩 끄고 켜며 비교하자.

In [ ]:
# [GEE]
vis_rgb  = {"bands": ["B4", "B3", "B2"],  "min": 0, "max": 1600, "gamma": 1.1}
vis_swir = {"bands": ["B12", "B8", "B4"], "min": 0, "max": 3500, "gamma": 1.1}

Map2 = geemap.Map(center=CENTER, zoom=10, basemap="HYBRID")
Map2.add_layer(pre_img,  vis_rgb,  "발화 전 진색")
Map2.add_layer(post_img, vis_rgb,  "발화 후 진색")
Map2.add_layer(post_img, vis_swir, "발화 후 SWIR 합성 (탄 자리=갈색·붉은색)")
Map2

**코드 설명**
- `thumb()`(앞에서 만든 함수)로 발화 전·후 진색을 나란히 그리고 `01_rgb_pre_post.png`로 저장한다(왼쪽 폴더 아이콘에서 볼 수 있다).

In [ ]:
# [GEE]
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, img, t in zip(axes, [pre_img, post_img], ["발화 전 2025-03-14", "발화 후 2025-04-08"]):
    ax.imshow(thumb(img, vis_rgb)); ax.set_title(t, fontsize=13); ax.axis("off")
plt.tight_layout(); plt.savefig("01_rgb_pre_post.png", dpi=130, bbox_inches="tight"); plt.show()

---
## 섹션 5. 식생·산불 지수와 피해 등급

이 섹션에서 지수 세 개를 계산한다.
- **NDVI** (정규식생지수): 잎이 많고 건강할수록 크다. 산불 뒤 잎이 얼마나 사라졌는지 본다.
- **NBR** (정규탄화지수): 탄 흔적에 민감하다. 근적외(B8)와 단파적외(B12)의 차이로 계산한다.
- **dNBR** (탄화지수 변화량): 발화 전 NBR에서 발화 후 NBR을 뺀 값이다. 클수록 많이 탔다.

아래 식에서 B4 = 빨강(Red), B8 = 근적외(NIR), B12 = 단파적외(SWIR2)다.
위성 센서는 밴드마다 좁은 파장 범위의 빛만 통과시키는 **필터**를 쓴다. 필터의 중심 파장은 B4 **665 nm**, B8 **842 nm**, B12 **2,190 nm**다(ESA 공칭값).

$$NDVI=\frac{B8-B4}{B8+B4} \qquad NBR=\frac{B8-B12}{B8+B12} \qquad dNBR=NBR_{전}-NBR_{후}$$

| dNBR | 등급 |
|---|---|
| < −0.1 | 재생 |
| −0.1 ~ 0.1 | 피해 없음 |
| 0.1 ~ 0.27 | 경미 |
| 0.27 ~ 0.44 | 중간-낮음 |
| 0.44 ~ 0.66 | 중간-높음 |
| > 0.66 | 극심 |

(USGS 기준, UN-SPIDER 권장 실무 정리)

**코드 설명**
- `normalizedDifference([A, B])` = (A − B) / (A + B). NDVI는 `[B8, B4]`, NBR은 `[B8, B12]`로 계산한다.
- `dnbr`: 발화 전 NBR에서 발화 후 NBR을 뺀다(`subtract`).
- `severity`: dNBR 값을 위 표의 6개 구간으로 나눠 등급 번호 1~6을 붙인다(`where`).
- `show()`: Earth Engine이 색을 칠한 그림을 받아 그리고, 같은 색 규칙으로 **컬러바**를 단다.
- 그림 1은 발화 전·후 NDVI, 그림 2는 발화 전·후 NBR과 dNBR, 피해 등급이다. 발화 전·후 그림은 같은 색 범위를 써서 바로 비교할 수 있다.

In [ ]:
# [GEE]
ndvi_pre  = pre_img.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndvi_post = post_img.normalizedDifference(["B8", "B4"]).rename("NDVI")
nbr_pre   = pre_img.normalizedDifference(["B8", "B12"]).rename("NBR")
nbr_post  = post_img.normalizedDifference(["B8", "B12"]).rename("NBR")
dnbr      = nbr_pre.subtract(nbr_post).rename("dNBR")

DNBR_BINS  = [-2.0, -0.1, 0.10, 0.27, 0.44, 0.66, 3.0]
SEV_LABELS = ["재생", "피해 없음", "경미", "중간-낮음", "중간-높음", "극심"]
SEV_COLORS = ["#1a9850", "#d9ef8b", "#fee08b", "#fdae61", "#e34a33", "#7f0000"]

severity = ee.Image(0)
for i in range(6):
    severity = severity.where(dnbr.gte(DNBR_BINS[i]).And(dnbr.lt(DNBR_BINS[i + 1])), i + 1)
severity = severity.updateMask(dnbr.mask()).rename("severity").toInt().clip(ROI)

from matplotlib.colors import LinearSegmentedColormap
PAL_VEG  = ["d73027", "fdae61", "fee08b", "d9ef8b", "1a9850"]   # 낮음(빨강) → 높음(초록)
PAL_DNBR = ["1a9850", "ffffbf", "fdae61", "d7191c", "7f0000"]   # 변화 없음(초록) → 큰 피해(검붉음)

def show(ax, image, vmin, vmax, palette, title, label):
    # Earth Engine이 색을 칠한 그림을 받고, 같은 색 규칙으로 컬러바를 단다
    ax.imshow(thumb(image, {"min": vmin, "max": vmax, "palette": palette}))
    ax.set_title(title, fontsize=12); ax.axis("off")
    cmap = LinearSegmentedColormap.from_list("pal", ["#" + c for c in palette])
    plt.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin, vmax)),
                 ax=ax, fraction=0.035, pad=0.02, label=label)

# 그림 1: NDVI 발화 전·후 (같은 색 범위라 바로 비교할 수 있다)
fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
show(axes[0], ndvi_pre,  0, 0.7, PAL_VEG, "발화 전 NDVI (3/14)", "NDVI")
show(axes[1], ndvi_post, 0, 0.7, PAL_VEG, "발화 후 NDVI (4/8)",  "NDVI")
plt.tight_layout(); plt.savefig("02_ndvi.png", dpi=130, bbox_inches="tight"); plt.show()

# 그림 2: NBR 발화 전·후 → dNBR → 피해 등급
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
show(axes[0, 0], nbr_pre,  -0.4, 0.6, PAL_VEG, "발화 전 NBR (3/14)", "NBR")
show(axes[0, 1], nbr_post, -0.4, 0.6, PAL_VEG, "발화 후 NBR (4/8)",  "NBR")
show(axes[1, 0], dnbr, -0.2, 0.8, PAL_DNBR, "dNBR = 발화 전 NBR - 발화 후 NBR", "dNBR")
ax = axes[1, 1]   # 피해 등급: 6칸짜리 컬러바에 등급 이름을 단다
ax.imshow(thumb(severity, {"min": 1, "max": 6, "palette": [c[1:] for c in SEV_COLORS]}))
ax.set_title("피해 등급 (dNBR을 6개 구간으로)", fontsize=12); ax.axis("off")
cb = plt.colorbar(plt.cm.ScalarMappable(cmap=ListedColormap(SEV_COLORS), norm=BoundaryNorm(np.arange(0.5, 7.5), 6)),
                  ax=ax, fraction=0.035, pad=0.02, ticks=range(1, 7))
cb.ax.set_yticklabels(SEV_LABELS)
plt.tight_layout(); plt.savefig("02_nbr_dnbr.png", dpi=130, bbox_inches="tight"); plt.show()

**코드 설명**
- `ee.Image.pixelArea()`: 픽셀마다 그 픽셀의 면적(m²)을 값으로 가진 영상이다.
- 면적 영상에 등급 영상을 붙이고(`addBands`), 등급별로 면적을 더한다(`Reducer.sum().group`). 30 m 격자로 서버에서 한 번에 계산한다.
- m²를 1만으로 나눠 ha로 바꾼다.

In [ ]:
# [GEE]
# 등급별 면적: 한 번의 요청으로 계산 (30 m 해상도)
area_stats = (ee.Image.pixelArea().addBands(severity)
              .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="cls"),
                            geometry=ROI, scale=30, maxPixels=1e10)
              .getInfo())
areas = {SEV_LABELS[int(g["cls"]) - 1]: g["sum"] / 1e4 for g in area_stats["groups"] if int(g["cls"]) >= 1}
for k in SEV_LABELS:
    print(f"  {k:8s}: {areas.get(k, 0):10,.0f} ha")
print(f"\n중간 이상(dNBR>0.27) 합계: {sum(areas.get(k, 0) for k in SEV_LABELS[3:]):,.0f} ha")
print(f"경미 포함(dNBR>0.1) 합계: {sum(areas.get(k, 0) for k in SEV_LABELS[2:]):,.0f} ha")

**생각해 보기**
- 공식 피해면적(집계 시점마다 다름)과 위 숫자를 비교해 보자. 어느 등급부터 '피해'에 포함될까?
- '경미' 등급에는 어떤 잡음이 섞일 수 있을까? (계절 변화, 잔설, 농경지 경운…)

---
## 섹션 6. AI로 산불 피해 찾기

### 왜 AI인가?
dNBR은 **발화 전·후 영상 두 장이 모두 깨끗해야** 계산된다. 실제로는 그렇지 않을 때가 많다.
- 발화 전 영상이 구름·잔설·황사로 덮였을 때 (봄 산불철에 흔하다. 이 실습의 3/14 영상에도 잔설이 있다 → 섹션 7-3, 7-4)
- 맑은 발화 전 영상이 몇 달 전 것뿐이라 계절이 맞지 않을 때
- 불이 꺼진 직후 첫 영상 한 장으로 빨리 지도를 만들어야 할 때

그래서 **발화 후 영상 한 장만 보고** 탄 곳을 찾는 AI를 만든다. AI는 dNBR 값을 계산하지 않는다. 픽셀마다 '탔다(1) / 안 탔다(0)'의 확률을 낸다.
정답은 현장 조사 대신 **두 장이 모두 깨끗한 곳의 dNBR**로 만든다. 사람이 매긴 정답이 아니라 지수로 자동으로 매긴 정답이다.

### 순서
**발화 전 영상은 정답(dNBR)을 만들 때만 쓴다. AI 입력은 모두 발화 후 영상 한 장에서 나온다.**
- MLP: 픽셀마다 밴드 6개(가시광 3 · 근적외 1 · 단파적외 2)와 이 밴드로 계산한 지수 4개(NDVI · NBR · NDWI · EVI), 모두 숫자 10개
- U-Net: 밴드 6개를 겹친 영상 조각(64 × 64 픽셀)

| 단계 | 장소 | 입력 | 정답·비교 대상 | 셀 |
|---|---|---|---|---|
| 정답 만들기 | 노란·빨간 상자 | (AI 안 씀) | 발화 전 + 발화 후 → dNBR → 피해(1)·정상(0) | 6-3 |
| 기준선 | 노란 → 빨간 상자 | 발화 후 NBR 하나 | 6-3의 정답 | 6-4 |
| 학습 | 노란 상자 A~D | 발화 후 영상 | 6-3의 정답 | 6-5, 6-7 |
| 검증 | 빨간 상자 (학습에 안 씀) | 발화 후 영상 | 6-3의 정답 ('흉내 점수') | 6-5 ~ 6-7 |
| 독립 확인 | 빨간 상자 | (검증 결과) | FIRMS 화점 | 7-2 |
| 쓰임 보기 | 잔설 상자 (발화 전 영상이 나쁜 곳) | 발화 후 영상 | dNBR, FIRMS | 7-4 |

**학습 데이터 한 줄은 이렇게 생겼다** (학습 상자 A에서 실제로 꺼낸 픽셀 두 개)

| | **입력**: 4/8 발화 후 영상, 그 픽셀의 밴드 6개 (반사율) | **정답** |
|---|---|---|
| 탄 픽셀 | B2 0.033 · B3 0.039 · B4 0.049 · B8 0.073 · B11 0.166 · B12 0.176 | **1 (피해)** ← dNBR +0.89 |
| 안 탄 픽셀 | B2 0.112 · B3 0.155 · B4 0.198 · B8 0.280 · B11 0.349 · B12 0.291 | **0 (정상)** ← dNBR +0.03 |

- 3/14 발화 전 영상은 오른쪽 **정답** 칸(dNBR)을 정할 때만 쓰인다. 왼쪽 **입력** 칸에는 3/14 값이 하나도 없다.
- 표에는 밴드 6개만 적었다. MLP는 이 밴드로 계산한 지수 4개를 더해 숫자 10개를 입력으로 쓴다(6-2).
- 학습 데이터는 이런 줄 12만 개짜리 표다. AI는 "왼쪽 숫자를 보면 오른쪽이 1인지 0인지"를 배운다.
- 탄 픽셀은 근적외(B8)가 낮고 단파적외(B12)가 높다. 4/8 영상 한 장만 봐도 이 생김새가 드러난다.
- 비유: 선생님이 사진 두 장(전·후)을 비교해 답안지를 매겨 두고, 학생(AI)은 사진 한 장(후)만 보고 답을 맞히는 연습을 한다.

```
MLP   : [픽셀 하나: 밴드 6개 + 지수 4개]  →  그 픽셀의 피해 확률 (0~1)
U-Net : [64×64 조각: 밴드 6개]           →  조각 전체의 피해 확률 지도
```

**정답 만드는 규칙**: dNBR > 0.27 → 피해(1), −0.1 < dNBR < 0.1 → 정상(0), 그 사이는 애매해서 뺀다.
정답은 dNBR **값**이 아니라 **class(1 = 피해, 0 = 정상)** 다. dNBR이 +0.89든 +0.30이든 정답은 똑같이 1이다. 그래서 MLP·U-Net은 **이진 분류** 모델이다.
이진 분류 모델은 마지막에 '1일 확률'(0~1)을 내고, 확률 > 0.5를 피해로 자른다.

**학습과 검증을 다른 곳에서**: 학습 상자 4개(노랑)로 배우고, 한 번도 보지 않은 검증 영역(빨강)에서 채점한다.
같은 영역에서 무작위로 나눠 채점하면 이웃 픽셀끼리 닮아서 점수가 부풀려진다.

### 6-1. 영상 내려받기: 검증 영역 + 학습 상자 4개
UTM 52N 좌표계의 **30 m 격자**로 잘라 GeoTIFF로 저장한다. 픽셀 하나 = 900 m² = 0.09 ha.
파일 11개를 받아서 몇 분 걸린다.

**코드 설명**
- `ee_to_geotiff()`: Earth Engine 영상을 UTM 52N 좌표계의 30 m 격자에 맞춰 GeoTIFF 파일로 받는다.
  - 영역의 UTM 좌표 범위를 구하고(`bounds`) 30 m 배수로 맞춘다. `bounds()`에 UTM 좌표계를 꼭 넘긴다. 안 넘기면 경위도가 나와 격자가 틀어진다.
  - `ee.data.computePixels`: 정해 준 격자 그대로 픽셀 값을 받아 온다.
  - 값이 없는 곳(구름, 영역 밖)은 −9999(`NODATA`)로 채운다.
  - 실패하면 `geemap`으로 한 번 더 시도한다.
- 검증 영역은 발화 전 영상·발화 후 영상·dNBR 3개, 학습 상자는 발화 후 영상·dNBR 2개씩 받는다. 모두 11개이고 2~3분 걸린다.
- 파일은 `/content/data`에 저장된다(왼쪽 폴더 아이콘). 런타임이 끊기면 사라진다.

In [ ]:
# [GEE]
DATA_DIR = "/content/data"; os.makedirs(DATA_DIR, exist_ok=True)
PRE_FILE, POST_FILE, DNBR_FILE = [f"{DATA_DIR}/{n}.tif" for n in ["pre", "post", "dnbr"]]
CRS, SCALE, NODATA = "EPSG:32652", 30, -9999

def ee_to_geotiff(image, region, path, scale=SCALE, crs=CRS):
    # 영역을 UTM 30 m 격자에 맞춰 GeoTIFF로 저장
    # bounds()에 좌표계를 안 주면 위경도가 나온다 → UTM 좌표계를 꼭 넘긴다
    b = region.bounds(1, ee.Projection(crs)).coordinates().getInfo()[0]
    xs, ys = [p[0] for p in b], [p[1] for p in b]
    x0, x1 = math.floor(min(xs) / scale) * scale, math.ceil(max(xs) / scale) * scale
    y0, y1 = math.floor(min(ys) / scale) * scale, math.ceil(max(ys) / scale) * scale
    req = {"expression": image.toFloat().unmask(NODATA),
           "fileFormat": "GEO_TIFF",
           "grid": {"dimensions": {"width": int((x1 - x0) / scale), "height": int((y1 - y0) / scale)},
                    "affineTransform": {"scaleX": scale, "shearX": 0, "translateX": x0,
                                        "shearY": 0, "scaleY": -scale, "translateY": y1},
                    "crsCode": crs}}
    try:
        data = ee.data.computePixels(req)
        with open(path, "wb") as f:
            f.write(data)
    except Exception as e:   # 요청이 너무 크거나 API가 다르면 geemap으로 다시 시도
        print("  computePixels 실패, geemap으로 재시도:", str(e)[:120])
        geemap.download_ee_image(image.toFloat().unmask(NODATA), path, region=region,
                                 crs=crs, scale=scale, dtype="float32")
    print("  저장:", path)

print("[검증 영역]")
ee_to_geotiff(pre_img.select(BANDS),  ROI_DL, PRE_FILE)
ee_to_geotiff(post_img.select(BANDS), ROI_DL, POST_FILE)
ee_to_geotiff(dnbr,                   ROI_DL, DNBR_FILE)

# 학습 상자는 발화 후 영상(입력)과 dNBR(정답)만 있으면 된다
TRAIN_FILES = {}
for name, box in TRAIN_BOXES.items():
    print(f"[학습 상자 {name}]")
    TRAIN_FILES[name] = (f"{DATA_DIR}/train_{name}_post.tif", f"{DATA_DIR}/train_{name}_dnbr.tif")
    ee_to_geotiff(post_img.select(BANDS), ee.Geometry.Rectangle(box), TRAIN_FILES[name][0])
    ee_to_geotiff(dnbr,                   ee.Geometry.Rectangle(box), TRAIN_FILES[name][1])

**코드 설명**
- `load_tif()`: GeoTIFF를 numpy 배열(밴드, 행, 열)로 읽는다.
- `valid`: 쓸 수 있는 픽셀은 True. NODATA가 아니고, dNBR이 숫자이고, 근적외(B8) 값이 0보다 커야 한다.
- 학습 상자도 같은 방식으로 읽어 `train_sets`에 담는다.
- 출력 읽기: 상자마다 크기, 유효 픽셀 비율, dNBR > 0.27(중간 이상 피해) 비율이 나온다.

In [ ]:
def load_tif(path):
    with rasterio.open(path) as src:
        return src.read().astype(np.float32), src.profile

pre_data,  profile = load_tif(PRE_FILE)
post_data, _       = load_tif(POST_FILE)
dnbr_data, _       = load_tif(DNBR_FILE)

valid = ((pre_data != NODATA).all(0) & (post_data != NODATA).all(0) &
         (dnbr_data[0] != NODATA) & np.isfinite(dnbr_data[0]) & (pre_data[3] > 0) & (post_data[3] > 0))
H, W = valid.shape
PIXEL_HA = (SCALE * SCALE) / 1e4
print(f"검증 영역: {H} × {W} 픽셀 ({H*W:,}개), 유효 {valid.mean()*100:.1f}%")
print(f"발화 후 B8(근적외) 평균 반사율: {post_data[3][valid].mean()/10000:.3f}")

train_sets = {}   # 이름 → (발화 후 영상, dNBR, 유효 마스크)
for name, (fp, fd) in TRAIN_FILES.items():
    post_k, dn_k = load_tif(fp)[0], load_tif(fd)[0][0]
    ok = (post_k != NODATA).all(0) & (dn_k != NODATA) & np.isfinite(dn_k) & (post_k[3] > 0)
    train_sets[name] = (post_k, dn_k, ok)
    print(f"학습 상자 {name}: {ok.shape[0]} × {ok.shape[1]}, 유효 {ok.mean()*100:.1f}%, dNBR>0.27 {((dn_k > 0.27) & ok).mean()*100:.0f}%")

### 6-2. 피처 만들기: 픽셀 하나 = 숫자 10개

**코드 설명**
- `spectral_indices()`: 밴드 6개로 지수 4개(NDVI, NBR, NDWI, EVI)를 계산한다. 값이 반사율 × 10000이라 먼저 10000으로 나눈다. `EPS`는 0으로 나누는 것을 막는 아주 작은 수다.
- 발화 후 영상의 피처 10개(밴드 6 + 지수 4)를 그림 10장으로 그린다. AI에게 픽셀 하나는 이 10개 숫자다.
- 색 범위는 값의 위·아래 2%를 잘라 대비를 높였다.

In [ ]:
EPS = 1e-10
def spectral_indices(data, scale=10000.0):
    b2, b3, b4, b8, b11, b12 = [data[i] / scale for i in range(6)]
    ndvi = (b8 - b4)  / (b8 + b4  + EPS)
    nbr  = (b8 - b12) / (b8 + b12 + EPS)
    ndwi = (b3 - b8)  / (b3 + b8  + EPS)
    evi  = 2.5 * (b8 - b4) / (b8 + 6 * b4 - 7.5 * b2 + 1 + EPS)
    return np.stack([ndvi, nbr, ndwi, evi])

IDX_NAMES = ["NDVI", "NBR", "NDWI", "EVI"]
pre_idx, post_idx = spectral_indices(pre_data), spectral_indices(post_data)
FEATURE_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12"] + IDX_NAMES

fig, axes = plt.subplots(2, 5, figsize=(18, 6.5))
feats_vis = np.vstack([post_data / 10000.0, post_idx])
for ax, f, name in zip(axes.flat, feats_vis, FEATURE_NAMES):
    v = f.copy(); v[~valid] = np.nan
    lo, hi = np.nanpercentile(v, [2, 98])
    im = ax.imshow(v, cmap="gray" if name.startswith("B") else "RdYlGn", vmin=lo, vmax=hi)
    ax.set_title(name); ax.axis("off")
plt.suptitle("발화 후 영상의 피처 10개", fontsize=13); plt.tight_layout()
plt.savefig("03_features.png", dpi=120, bbox_inches="tight"); plt.show()

### 6-3. 정답 만들기: dNBR로 피해(1)·정상(0) 매기기

**코드 설명**
- `make_labels()`: dNBR > 0.27이면 피해, −0.1 < dNBR < 0.1이면 정상이다. 그 사이(0.1~0.27)는 애매해서 학습에서 뺀다.
- `pixel_features()`: 영상(6, 행, 열)을 표(픽셀 수, 10)로 편다. 표의 한 줄이 픽셀 하나, 열 10개가 피처 10개다.
- **검증 데이터**: 빨간 상자의 피해·정상 픽셀 전부.
- **학습 데이터**: 노란 상자 4개에서 피해 6만, 정상 6만 픽셀을 무작위로 뽑는다. 수를 맞추는 이유는 정상 픽셀이 훨씬 많기 때문이다. 그대로 쓰면 AI가 "다 정상"이라고만 해도 점수가 높게 나온다.
- `default_rng(42)`: 무작위 뽑기의 시작값을 고정해, 누가 돌려도 같은 픽셀이 뽑힌다.

In [ ]:
BURN_T, NORMAL_T = 0.27, 0.10

def make_labels(dn, ok):        # dNBR → 피해 마스크, 정상 마스크
    return (dn > BURN_T) & ok, (dn < NORMAL_T) & (dn > -0.1) & ok

def pixel_features(post):       # (6, H, W) 영상 → (픽셀 수, 10) 표
    f = np.vstack([(post / 10000.0).reshape(6, -1), spectral_indices(post).reshape(4, -1)]).T
    return np.nan_to_num(f.astype(np.float32))

# 검증 = 빨간 상자 ROI_DL (학습에 쓰지 않는다)
d = dnbr_data[0]
burn_mask, normal_mask = make_labels(d, valid)
all_feats = pixel_features(post_data)
va_idx = np.flatnonzero((burn_mask | normal_mask).ravel())
X_va, y_va = all_feats[va_idx], burn_mask.ravel()[va_idx].astype(np.float32)

# 학습 = 상자 4개에서 피해·정상 픽셀을 같은 수만큼 뽑는다
burn_px, normal_px = [], []
for post_k, dn_k, ok in train_sets.values():
    b, n = make_labels(dn_k, ok); f = pixel_features(post_k)
    burn_px.append(f[b.ravel()]); normal_px.append(f[n.ravel()])
burn_px, normal_px = np.vstack(burn_px), np.vstack(normal_px)

rng = np.random.default_rng(42)
n_sample = int(min(len(burn_px), len(normal_px), 60000))
X_tr = np.vstack([burn_px[rng.choice(len(burn_px), n_sample, replace=False)],
                  normal_px[rng.choice(len(normal_px), n_sample, replace=False)]])
y_tr = np.hstack([np.ones(n_sample), np.zeros(n_sample)]).astype(np.float32)
perm = rng.permutation(len(X_tr)); X_tr, y_tr = X_tr[perm], y_tr[perm]

print(f"학습 (상자 4개): 피해 후보 {len(burn_px):,} / 정상 후보 {len(normal_px):,} 픽셀 → {len(X_tr):,}개 (클래스 균형)")
print(f"검증 영역 (학습에 안 씀): {len(X_va):,}개 (피해 {int(y_va.sum()):,} / 정상 {len(y_va) - int(y_va.sum()):,})")

### 6-4. 기준선: AI 없이 발화 후 NBR 하나로
AI가 정말 필요한지 보려면 비교할 기준이 있어야 한다. 발화 후 영상 한 장만 쓰는 가장 단순한 방법은 **발화 후 NBR에 문턱값 하나**를 거는 것이다.
학습 상자에서 피해·정상을 가장 잘 가르는 문턱값을 고르고, 검증 영역에서 AI와 똑같이 채점한다.

**코드 설명**
- `post_nbr()`: 발화 후 영상의 NBR = (B8 − B12) / (B8 + B12).
- 문턱값 고르기: −0.5부터 0.6까지 221개 값을 차례로 넣어 본다. '피해 픽셀 중 문턱값 아래인 비율'과 '정상 픽셀 중 문턱값 이상인 비율'의 평균이 가장 큰 값을 고른다. 학습 상자(노랑)의 정답만 쓴다.
- 채점: 검증 영역(빨강)에서 'NBR < 문턱값'을 피해로 보고, MLP·U-Net과 같은 픽셀·같은 정답으로 정확도를 잰다.
- 출력 읽기: 이 정확도가 AI가 넘어야 할 기준선이다. 발화 후 NBR이 낮은 곳은 탄 땅만이 아니다(맨땅, 마을, 그늘진 비탈 등). 그래서 문턱값 하나로는 헷갈린다.

In [ ]:
def post_nbr(post):          # 발화 후 영상의 NBR
    b8, b12 = post[3] / 10000.0, post[5] / 10000.0
    return (b8 - b12) / (b8 + b12 + EPS)

# 학습 상자에서 문턱값 고르기
burn_v, normal_v = [], []
for post_k, dn_k, ok in train_sets.values():
    b, n = make_labels(dn_k, ok); v = post_nbr(post_k)
    burn_v.append(v[b]); normal_v.append(v[n])
burn_v, normal_v = np.concatenate(burn_v), np.concatenate(normal_v)
cands = np.linspace(-0.5, 0.6, 221)
score = [((burn_v < t).mean() + (normal_v >= t).mean()) / 2 for t in cands]
NBR_T = cands[int(np.argmax(score))]

# 검증 영역에서 채점 (AI와 같은 픽셀, 같은 정답)
lab, v = burn_mask | normal_mask, post_nbr(post_data)
thr_val_acc = ((v[lab] < NBR_T) == burn_mask[lab]).mean() * 100
thr_burn = (v < NBR_T) & valid
print(f"문턱값: 발화 후 NBR < {NBR_T:.2f}")
print(f"검증 영역 정확도: {thr_val_acc:.1f}%  |  피해 면적: {thr_burn.sum()*PIXEL_HA:,.0f} ha")

### 6-5. 학습: MLP, 픽셀 하나만 보는 모델
```
입력 10 → 64 (BatchNorm·ReLU·Dropout 0.3) → 32 (BatchNorm·ReLU·Dropout 0.2) → 16 (ReLU) → 1 (Sigmoid)
```

**코드 설명**
- `BurnAreaMLP`: 입력 10 → 64 → 32 → 16 → 출력 1로 이어지는 신경망이다.
  - `Linear`: 입력에 가중치를 곱해 더하는 층. 학습으로 맞추는 숫자가 여기 있다.
  - `BatchNorm1d`: 값의 크기를 고르게 맞춰 학습을 안정시킨다.
  - `ReLU`: 음수는 0으로 바꾼다. 곡선 같은 관계를 배울 수 있게 해 준다.
  - `Dropout`: 학습할 때 연결 일부를 무작위로 끊어, 학습 자료를 통째로 외우는 것을 막는다.
  - `Sigmoid`: 마지막 값을 0~1 사이 확률로 바꾼다.
- `torch.manual_seed(42)`: 처음 가중치를 고정해 결과를 재현할 수 있게 한다.
- 파라미터 수 3,521개는 AI가 학습으로 맞추는 숫자의 개수다.

In [ ]:
class BurnAreaMLP(nn.Module):
    def __init__(self, in_features=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x):
        return self.net(x).squeeze(-1)

torch.manual_seed(42)
mlp = BurnAreaMLP().to(device)
print(mlp); print("파라미터 수:", sum(p.numel() for p in mlp.parameters()))

**코드 설명**
- `DataLoader`: 학습 데이터를 섞어서 4,096개씩 묶음(batch)으로 넣는다.
- `BCELoss`: 예측 확률이 정답(1 = 피해, 0 = 정상)과 얼마나 다른지 재는 손실 함수다.
- `Adam`: 손실이 줄어드는 쪽으로 가중치를 고치는 방법이다. `lr`(학습률)은 한 번에 고치는 크기다.
- `CosineAnnealingLR`: 학습률을 에폭마다 조금씩 줄여 끝에서 안정되게 한다.
- **에폭(epoch)**: 학습 데이터를 한 바퀴 다 보는 것. 30번 돈다.
- 에폭마다 검증 영역에서 정확도를 잰다. 확률 > 0.5를 피해로 보고 정답과 같은 비율이다.
- 그래프 읽기: 손실이 내려가고 정확도가 올라가면 학습이 되는 것이다. 학습 정확도만 오르고 검증 정확도가 떨어지면 외우기(과적합)다.
- **학습 정확도가 검증 정확도보다 낮게 나와도 괜찮다.** 두 숫자는 재는 조건이 다르다.
  - 학습 정확도는 학습하는 도중(`mlp.train()`)에 잰다. 이때 `Dropout`이 연결 일부를 일부러 끊어 두고, 한 에폭 안에서도 가중치가 계속 바뀐다. 출력되는 값은 그 묶음(batch)들의 평균이다.
  - 검증 정확도는 에폭이 끝난 뒤 `Dropout`을 끄고(`mlp.eval()`) 완성된 모델로 잰다.
  - 학습 데이터(상자 4곳에서 피해·정상을 같은 수로 뽑은 픽셀)와 검증 데이터(검증 영역의 정답 픽셀 전부)는 픽셀 구성도 다르다.
  - 그래서 믿을 숫자는 검증 정확도다. 검증이 학습보다 **크게 낮을 때**가 문제다(학습 상자를 외운 것).

In [ ]:
loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)), batch_size=4096, shuffle=True)
Xva_t, yva_t = torch.from_numpy(X_va).to(device), torch.from_numpy(y_va).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 30
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
hist = {"loss": [], "train_acc": [], "val_acc": []}

for epoch in range(EPOCHS):
    mlp.train(); tot, correct, loss_sum = 0, 0, 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(); p = mlp(xb); loss = criterion(p, yb); loss.backward(); optimizer.step()
        loss_sum += loss.item() * len(yb); correct += ((p > 0.5) == (yb > 0.5)).sum().item(); tot += len(yb)
    scheduler.step()
    mlp.eval()
    with torch.no_grad():
        va = ((mlp(Xva_t) > 0.5) == (yva_t > 0.5)).float().mean().item()
    hist["loss"].append(loss_sum / tot); hist["train_acc"].append(correct / tot * 100); hist["val_acc"].append(va * 100)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss {hist['loss'][-1]:.4f}  학습 {hist['train_acc'][-1]:.1f}%  검증 {hist['val_acc'][-1]:.1f}%")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(hist["loss"]); a1.set_title("학습 손실"); a1.set_xlabel("Epoch"); a1.grid(alpha=.3)
a2.plot(hist["train_acc"], label="학습 (상자 4개)"); a2.plot(hist["val_acc"], label="검증 영역"); a2.set_title("정확도 (%)")
a2.set_xlabel("Epoch"); a2.legend(); a2.grid(alpha=.3)
plt.tight_layout(); plt.savefig("04_training.png", dpi=120, bbox_inches="tight"); plt.show()

> 검증 정확도는 **학습에 쓰지 않은 검증 영역**에서 잰 값이다. 그래도 **정답이 dNBR에서 나왔으므로** 이 숫자는 "MLP가 처음 보는 곳에서도 dNBR 규칙을 잘 흉내 낸다"는 뜻일 뿐, "산불을 잘 찾는다"는 뜻은 아니다. 섹션 7에서 독립된 자료로 다시 확인한다.

### 6-6. 검증 영역 전체에 MLP 적용

**코드 설명**
- 검증 영역의 모든 픽셀(약 44만 개)을 5만 개씩 MLP에 넣어 피해 확률을 구한다. `torch.no_grad()`는 학습 없이 예측만 한다는 뜻이다.
- 표로 펼쳤던 확률을 다시 지도 모양(행, 열)으로 되돌린다(`reshape`).
- 확률 > 0.5인 픽셀을 피해로 보고 면적을 계산한다. 픽셀 하나는 0.09 ha다.
- MLP가 내는 값은 0/1이 아니라 **피해일 확률(0~1)** 이다. 이진 분류 모델은 확률을 내고, 0.5에서 잘라 피해(1)·정상(0)을 정한다. 확률 지도는 7-1에서 본다.
- 마지막 줄은 dNBR 피해 지도와 MLP 피해 지도가 같은 판정을 내린 픽셀의 비율이다.

In [ ]:
mlp.eval(); prob = np.zeros(H * W, np.float32)
with torch.no_grad():
    for i in range(0, H * W, 50000):
        prob[i:i + 50000] = mlp(torch.from_numpy(all_feats[i:i + 50000]).to(device)).cpu().numpy()
prob_map = prob.reshape(H, W); prob_map[~valid] = np.nan
mlp_burn = (prob_map > 0.5) & valid
dnbr_burn = (d > BURN_T) & valid
print(f"MLP 피해(확률>0.5): {mlp_burn.sum()*PIXEL_HA:,.0f} ha")
print(f"dNBR 피해(>0.27):   {dnbr_burn.sum()*PIXEL_HA:,.0f} ha")
print(f"두 지도의 일치 (같은 판정 비율): {((mlp_burn == dnbr_burn) & valid).sum() / valid.sum() * 100:.1f}%")

### 6-7. U-Net: 주변까지 보는 모델 (CNN)
MLP는 픽셀 하나만 본다. 그런데 위성영상은 **영상**이라 이웃 픽셀이 만드는 모양·경계에도 정보가 있다.
U-Net(CNN의 한 종류)은 64×64 조각 전체를 보고 이것까지 학습한다.
MLP와 똑같이 **학습 상자 4개의 조각으로 배우고**, 에폭마다 검증 영역에서 채점한다.
학습률은 에폭마다 조금씩 줄인다(cosine). 마지막 에폭에서 점수가 출렁이지 않게 하려는 것이다.
GPU가 없으면 에폭이 자동으로 2로 줄어든다.

**코드 설명**
- `unet_input()` / `unet_label()`: 입력 영상(밴드 6개)과 정답 지도(1 = 피해, 0 = 정상, −1 = 학습 제외)를 만든다.
- **조각 자르기**: 노란 상자를 64×64 조각으로 자른다. 32픽셀씩 옮겨 가며 자르니 이웃 조각과 반씩 겹친다. 정답이 있는 픽셀이 30% 넘는 조각만 쓴다.
- `batch()`: 조각 16개를 한 묶음으로 만들고, 그중 절반은 좌우를 뒤집는다(증강). 같은 자료로 더 다양한 예를 보여 주는 방법이다.
- `smp.Unet("resnet18", ...)`: 앞부분(ResNet18)이 조각을 점점 줄이며 특징을 뽑고, 뒷부분이 다시 키우며 픽셀마다 확률을 낸다. `encoder_weights=None`은 미리 학습된 값 없이 처음부터 배운다는 뜻이고, `in_channels=6`은 밴드 6개다.
- 손실은 정답이 있는 픽셀(−1이 아닌 곳)에서만 잰다.
- 에폭마다 빨간 상자 전체를 한 번에 넣어 검증 정확도를 잰다. U-Net은 크기를 반씩 다섯 번 줄이므로 가로·세로를 32의 배수로 채워(패딩) 넣는다.
- 에폭 수는 GPU면 6, CPU면 자동으로 2다.
- 출력 읽기: 중간 에폭에서 검증 정확도가 크게 떨어졌다가 돌아올 수 있다. 학습률이 큰 앞쪽 에폭에서 흔들리는 것이고, 학습률이 줄어드는 뒤쪽 에폭에서 안정된다. 그래서 마지막 에폭의 값을 본다.

In [ ]:
import segmentation_models_pytorch as smp

PATCH, STRIDE = 64, 32
UNET_EPOCHS = 6 if device.type == "cuda" else 2   # CPU면 자동으로 줄인다

def unet_input(post, ok):       # (6, H, W) 반사율, 유효하지 않은 곳은 0
    x = np.nan_to_num(post / 10000.0).astype(np.float32); x[:, ~ok] = 0
    return x

def unet_label(dn, ok):         # 1 = 피해, 0 = 정상, -1 = 학습 제외
    b, n = make_labels(dn, ok)
    lb = np.full(dn.shape, -1, np.float32); lb[b] = 1; lb[n] = 0
    return lb

# 학습 조각: 상자 4개에서 32픽셀씩 옮기며 64×64로 자른다
tr_imgs = [(unet_input(p, ok), unet_label(dn, ok)) for p, dn, ok in train_sets.values()]
patches = []
for k, (_, lb) in enumerate(tr_imgs):
    for r in range(0, lb.shape[0] - PATCH + 1, STRIDE):
        for c in range(0, lb.shape[1] - PATCH + 1, STRIDE):
            if (lb[r:r + PATCH, c:c + PATCH] >= 0).mean() > 0.3:
                patches.append((k, r, c))
print("학습 조각 수 (상자 4개):", len(patches))

def batch(idx):
    xs = np.stack([tr_imgs[k][0][:, r:r + PATCH, c:c + PATCH] for k, r, c in idx])
    ys = np.stack([tr_imgs[k][1][r:r + PATCH, c:c + PATCH] for k, r, c in idx])[:, None]
    flip = rng.random(len(idx)) < 0.5                 # 좌우 반전 증강
    xs[flip] = xs[flip][..., ::-1]; ys[flip] = ys[flip][..., ::-1]
    return torch.from_numpy(xs.copy()).to(device), torch.from_numpy(ys.copy()).to(device)

# 검증 영역 전체를 한 번에 넣는다 (32의 배수로 패딩)
X_img, label = unet_input(post_data, valid), unet_label(d, valid)
ph, pw = (32 - H % 32) % 32, (32 - W % 32) % 32
x_full = torch.from_numpy(np.pad(X_img, ((0, 0), (0, ph), (0, pw)), mode="reflect")[None]).to(device)
def unet_predict():
    unet.eval()
    with torch.no_grad():
        return torch.sigmoid(unet(x_full))[0, 0, :H, :W].cpu().numpy()

torch.manual_seed(42)
unet = smp.Unet("resnet18", encoder_weights=None, in_channels=6, classes=1).to(device)
opt = optim.AdamW(unet.parameters(), lr=1e-3)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=UNET_EPOCHS)   # 학습률을 에폭마다 줄인다
bce = nn.BCEWithLogitsLoss(reduction="none")
for ep in range(UNET_EPOCHS):
    unet.train(); order = rng.permutation(len(patches)); tot = 0.0
    for i in range(0, len(order), 16):
        xb, yb = batch([patches[j] for j in order[i:i + 16]])
        m = (yb >= 0).float()
        loss = (bce(unet(xb), yb.clamp(min=0)) * m).sum() / m.sum().clamp(min=1)
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
    sched.step()
    p, m = unet_predict(), label >= 0
    unet_val_acc = ((p[m] > 0.5) == (label[m] > 0.5)).mean() * 100
    print(f"U-Net epoch {ep+1}/{UNET_EPOCHS}  loss {tot / max(1, len(order) // 16):.4f}  검증 영역 {unet_val_acc:.1f}%")

unet_prob = unet_predict(); unet_prob[~valid] = np.nan
unet_burn = (unet_prob > 0.5) & valid
print(f"U-Net 피해(확률>0.5): {unet_burn.sum()*PIXEL_HA:,.0f} ha")

---
## 섹션 7. 검증: 정말 산불일까?

### 7-1. 검증 영역 결과 한눈에 보기

**코드 설명**
- `rgb()`: 밴드 B4·B3·B2를 0~1 범위로 바꿔 진색 그림을 만든다.
- `dnbr_cls`: 검증 영역의 dNBR을 6개 등급으로 나눈 지도다.
- 여섯 장을 그린다. 위 줄은 발화 전·후 진색과 dNBR 등급(정답이 나온 곳)이다. 아래 줄은 기준선(NBR 문턱값), MLP, U-Net이다.
- **아래 줄 세 장은 모두 발화 후 영상 한 장만 보고 낸 결과다.**
- 기준선은 0/1 지도라 컬러바가 '0 정상 / 1 피해' 두 칸이다. MLP·U-Net은 **확률(0~1)** 지도라 컬러바가 초록(0) → 노랑(0.5) → 빨강(1)으로 이어진다.

In [ ]:
def rgb(data, hi=1600):
    out = np.clip(np.stack([data[2], data[1], data[0]], -1) / hi, 0, 1); out[~valid] = 1; return out

dnbr_cls = np.full((H, W), np.nan)
for i, (lo, hi) in enumerate(zip(DNBR_BINS[:-1], DNBR_BINS[1:])):
    dnbr_cls[(d >= lo) & (d < hi) & valid] = i

PROB = dict(cmap="RdYlGn_r", vmin=0, vmax=1)
CLASS = dict(cmap=ListedColormap(["#006837", "#a50026"]), norm=BoundaryNorm([-0.5, 0.5, 1.5], 2), interpolation="nearest")
SEV = dict(cmap=ListedColormap(SEV_COLORS), norm=BoundaryNorm(np.arange(-0.5, 6.5), 6), interpolation="nearest")
panels = [("발화 전 진색", rgb(pre_data), {}, None), ("발화 후 진색", rgb(post_data), {}, None),
          ("dNBR 등급 (두 장, 정답이 나온 곳)", dnbr_cls, SEV, "sev"),
          ("기준선: 발화 후 NBR 문턱값 (0/1)", np.where(valid, thr_burn, np.nan), CLASS, "class"),
          ("MLP 피해 확률 (0.5 넘으면 피해)", prob_map, PROB, "prob")]
if "unet_prob" in globals():
    panels.append(("U-Net 피해 확률 (0.5 넘으면 피해)", unet_prob, PROB, "prob"))
fig, axes = plt.subplots(2, 3, figsize=(19, 9.5))
for ax in axes.flat: ax.axis("off")
for ax, (t, a, kw, cb) in zip(axes.flat, panels):
    im = ax.imshow(a, **kw); ax.set_title(t, fontsize=12)
    if cb == "prob":
        plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02, label="피해 확률")
    elif cb == "class":
        c = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02, ticks=[0, 1]); c.ax.set_yticklabels(["0 정상", "1 피해"])
    elif cb == "sev":
        c = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02, ticks=range(6)); c.ax.set_yticklabels(SEV_LABELS)
plt.tight_layout(); plt.savefig("05_comparison.png", dpi=120, bbox_inches="tight"); plt.show()

**그림 읽기: 0/1 지도와 확률 지도**
- 아래 줄의 **기준선**은 0/1 지도다. 정상은 초록, 피해는 빨강 두 색뿐이다.
- **MLP·U-Net**은 0/1이 아니라 **피해 확률(0~1)** 지도다. 면적을 셀 때는 확률 > 0.5를 피해로 자른다.
- **노란색(확률 0.5 근처)** 은 AI가 확신하지 못한 픽셀이다. 학습 때 뺀 애매한 픽셀(dNBR 0.1~0.27)과, 30 m 안에 탄 곳·안 탄 곳이 섞인 경계 픽셀에서 주로 나온다.

**생각해 보기**
- 기준선(문턱값) 지도에서 dNBR 정답과 다른 곳은 어디인가? 진색 영상에서 그곳이 무엇인지 보자.
- MLP 지도와 U-Net 지도의 **모양**을 비교하자. 어느 쪽이 점점이 튀고, 어느 쪽이 덩어리지고 경계가 매끈한가?
- 그런데 두 모델의 검증 정확도는 비슷하다. 왜일까?

> **정리: MLP와 U-Net은 무엇이 다른가**
> - **MLP는 픽셀 하나만 본다.** 주변이 모두 멀쩡해도 그 픽셀 하나가 탄 땅처럼 보이면 피해로 잡는다. 그래서 안 탄 지역에도 노란·빨간 점이 흩어지고 경계가 거칠다.
> - **U-Net은 64×64 이웃을 함께 본다.** 피해지가 덩어리로 이어지는 모양을 배워서, 외딴 점은 거의 만들지 않고 경계를 지형을 따라 매끈하게 그린다. 대신 아주 작은 피해 조각은 주변에 묻혀 놓칠 수 있다.
> - **검증 정확도가 비슷한 이유**: 정답(dNBR)은 픽셀 하나씩 따로 정해진 값이다. 그래서 이 채점으로는 '모양을 보는 능력'이 잘 드러나지 않는다. 차이는 지도 모양과, 7-4의 잔설 상자처럼 처음 보는 곳에서 더 잘 보인다.

### 7-2. 독립 자료와 비교: NASA FIRMS 화점
**왜 FIRMS와 비교하나?** 지금까지의 정확도는 dNBR로 만든 정답으로 채점한 값이다. dNBR이 틀린 곳(잔설, 구름, 계절 차이)이 있어도 정답과 채점이 함께 틀려서 점수에 드러나지 않는다.
그래서 **만드는 원리가 다른 자료**로 '불이 실제로 있었나'를 확인한다.
- FIRMS는 위성이 **불타는 순간의 열**을 감지한 기록이다. dNBR(타고 난 뒤의 반사율 변화)과 원리가 다르다.
- 다만 FIRMS도 정답은 아니다. Earth Engine의 `FIRMS`는 MODIS 기반이라 화소가 약 1 km로 거칠고, 위성이 지나가는 시각(하루 몇 번)에 타고 있던 곳만 잡는다. 연기·구름에 가리면 놓친다.
- 그래서 30 m 경계까지 맞추는 데는 쓰지 않고, **'그 근처에서 불이 났나'** 를 확인하는 데 쓴다. 7-4에서는 이것으로 불이 나지 않은 곳을 가려낸다.
- MODIS 격자(사인 도법)를 UTM 지도에 옮기면 화소가 **비스듬한 평행사변형**으로 보인다. 아래 그림의 사선 테두리는 오류가 아니다.

**코드 설명**
- `ImageCollection("FIRMS")`: NASA FIRMS 화점 자료(MODIS 위성, 약 1 km)다. `T21` 밴드는 화점의 밝기온도(K)이고, 화점이 잡힌 곳에만 값이 있다.
- 3월 22일~31일 중 한 번이라도 화점이 잡힌 곳은 1, 아닌 곳은 0으로 만든다(`max().gt(0)`, 빈 곳은 `unmask(0)`).
- 6-1과 같은 함수로 빨간 상자를 30 m 격자에 맞춰 받는다.

In [ ]:
# [GEE]
FIRE_FILE = f"{DATA_DIR}/firms.tif"
firms = (ee.ImageCollection("FIRMS").filterBounds(ROI_DL)
         .filterDate("2025-03-22", "2025-04-01").select("T21")
         .max().gt(0).unmask(0).rename("firms"))
ee_to_geotiff(firms, ROI_DL, FIRE_FILE)

**코드 설명**
- `fire`: 화점 영역(True/False).
- `distance_transform_edt`: 픽셀마다 가장 가까운 화점까지의 거리(픽셀 수)를 잰다. 30 m를 곱해 미터로 바꾼다.
- `near_fire`: 화점에서 1 km 안인 곳이다. FIRMS 화소가 약 1 km라 그만큼의 위치 오차를 봐준다.
- `near()`: 각 방법이 '피해'로 판정한 픽셀 중 화점에서 1 km 안에 있는 비율을 잰다. dNBR, 기준선(NBR 문턱값), MLP, U-Net을 같은 방식으로 잰다.
- 진색 위에 화점 경계(하늘색)와 MLP 피해 경계(빨강)를 선으로 그린다(`contour`).
- 출력 읽기: 피해 판정의 대부분이 화점 1 km 안에 있으면, 그 '피해'는 실제 산불이 난 곳과 맞는다고 볼 수 있다. 30 m 경계가 화점 테두리와 딱 맞지 않는 것은 FIRMS가 거칠기 때문이다.

In [ ]:
from scipy.ndimage import distance_transform_edt

fire = load_tif(FIRE_FILE)[0][0] > 0.5
fire &= valid
print(f"FIRMS 화점 영역: {fire.sum()*PIXEL_HA:,.0f} ha  (1 km 화소라 거칠다)")

dist_m = distance_transform_edt(~fire) * SCALE      # 가장 가까운 화점까지의 거리 (m)
near_fire = (dist_m <= 1000) & valid                # 화점에서 1 km 안

def near(pred, name):
    pct = (pred & near_fire).sum() / max(1, pred.sum()) * 100
    print(f"{name:10s} | 피해 판정 {pred.sum()*PIXEL_HA:7,.0f} ha 중 {pct:5.1f}%가 화점에서 1 km 안")
    return pct

for m, name in [(dnbr_burn, "dNBR"), (thr_burn, "NBR문턱값"), (mlp_burn, "MLP")]:
    near(m, name)
if "unet_burn" in globals():
    near(unet_burn, "U-Net")

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.imshow(rgb(post_data)); ax.contour(fire.astype(float), levels=[0.5], colors="cyan", linewidths=1)
ax.contour(mlp_burn.astype(float), levels=[0.5], colors="red", linewidths=0.6)
ax.set_title("FIRMS 화점 영역(하늘색) vs MLP 피해(빨강)"); ax.axis("off")
plt.tight_layout(); plt.savefig("06_firms_check.png", dpi=120, bbox_inches="tight"); plt.show()

**생각해 보기**
- 화점에서 1 km 밖인데 '피해'로 잡힌 곳은 무엇일까? 진색 영상으로 확인해 보자. (실제로 탔지만 위성이 지나갈 때 불이 없었던 곳, 논밭·맨땅처럼 산불이 아닌 변화…)
- FIRMS로 확인할 수 있는 것과 없는 것은 무엇일까? ('근처에서 불이 났나'는 확인된다. 30 m 경계는 FIRMS로 확인할 수 없다.)

### 7-3. 잔설이 만든 가짜 산불
피해지 북쪽 산지에 상자 하나(`SNOW_BOX`, 약 19 km × 12 km)를 잡는다. 이번 산불은 여기까지 오지 않았다(7-4에서 FIRMS 화점으로 확인한다).
발화 전 3/14 영상에는 이곳 능선에 **잔설**이 남아 있었고, 발화 후 4/8에는 녹았다.
눈은 근적외에서 밝고 단파적외에서 어두워 NBR이 높다. 눈이 녹으면 NBR이 떨어져 **불이 나지 않았는데도 dNBR이 커진다.**

먼저 위치를 보자. 잔설 상자는 학습 상자에도 검증 영역에도 들어가지 않은 곳이다.

**코드 설명** (위치 지도)
- 섹션 5의 dNBR 피해 등급 지도를 바탕에 깔고, 지금까지 쓴 상자를 모두 그린다. 빨강 = 검증 영역, 노랑 = 학습 상자 A~D, 하늘색 = 잔설 상자.
- `SNOW_XY`: 잔설 상자의 좌표(서쪽 경도, 남쪽 위도, 동쪽 경도, 북쪽 위도). 다음 셀에서 이 좌표로 상자(`SNOW_BOX`)를 만든다.
- `thumb_ll()`: 섹션 4에서 만든 함수다. 경위도 격자 그대로 그림을 받아 상자를 정확히 겹친다.
- 출력 읽기: 잔설 상자는 피해지 북쪽 산지에 있고, 학습에도 검증에도 쓰지 않은 곳이다. 7-1의 그림은 빨간 상자 안만 보여 줬다.
- 그런데 잔설 상자 안에도 dNBR '피해' 색(주황·빨강)이 보인다. 불이 여기까지 오지 않았는데 왜일까? 아래 셀에서 확인한다.

In [ ]:
# [GEE]
# 위치 지도: 검증 영역·학습 상자·잔설 상자가 어디에 있나 (바탕은 섹션 5의 dNBR 피해 등급)
SNOW_XY = [129.15, 36.64, 129.36, 36.75]          # 잔설 상자 (다음 셀에서 쓴다)

loc_xy = np.array(ROI.coordinates().getInfo()[0])
(lx0, ly0), (lx1, ly1) = loc_xy.min(0), loc_xy.max(0)
sev_img = severity.visualize(min=1, max=6, palette=[c.lstrip("#") for c in SEV_COLORS])

fig, ax = plt.subplots(figsize=(13, 7.5))
ax.imshow(thumb_ll(sev_img, ROI), extent=[lx0, lx1, ly0, ly1], aspect=1 / np.cos(np.radians(36.5)))

def draw_box(b, color, text, text_color):   # 상자 테두리와 이름표
    ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=color, lw=2.5))
    ax.text((b[0] + b[2]) / 2, (b[1] + b[3]) / 2, text, color=text_color, fontsize=13, fontweight="bold",
            ha="center", va="center", bbox=dict(fc=color, ec="none", alpha=0.9, boxstyle="round,pad=0.3"))

draw_box(ROI_DL_BOX, "#e8202a", "검증", "white")
for name, b in TRAIN_BOXES.items():
    draw_box(b, "#ffd400", f"학습 {name}", "black")
draw_box(SNOW_XY, "#00e5ff", "잔설", "black")
ax.set_xlim(lx0, lx1); ax.set_ylim(ly0, ly1); ax.set_xlabel("경도"); ax.set_ylabel("위도")
ax.set_title("위치 지도: 검증 영역(빨강) · 학습 상자(노랑) · 잔설 상자(하늘색) · 바탕은 dNBR 피해 등급", fontsize=12)
plt.tight_layout(); plt.savefig("07_location.png", dpi=110, bbox_inches="tight"); plt.show()

**코드 설명**
- `SNOW_BOX`: 잔설을 확인할 상자다. 좌표(`SNOW_XY`)는 위치 지도 셀에서 정했다.
- NDSI(정규화 눈 지수) = (B3 − B11) / (B3 + B11). 눈은 초록에서 밝고 단파적외에서 어두워 값이 크다. 0.4를 넘으면 눈으로 본다.
- 그림 4장: 발화 전(잔설), 발화 후, dNBR 등급, 눈 판정(NDSI > 0.4).
- 상자를 6-1과 같은 30 m 격자 파일 4개로 받는다. 발화 후 영상(7-4의 AI 입력), dNBR, 발화 전 NDSI, FIRMS 화점이다. **7-3과 7-4는 모두 이 파일로 계산**하므로 숫자가 서로 맞는다.

In [ ]:
# [GEE]
SNOW_BOX = ee.Geometry.Rectangle(SNOW_XY)   # 좌표는 위치 지도 셀에서 정했다
ndsi_pre = pre_img.normalizedDifference(["B3", "B11"]).rename("ndsi")   # 정규화 눈 지수 (발화 전)
snow = ndsi_pre.gt(0.4)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))
axes[0].imshow(thumb(pre_img,  vis_rgb, SNOW_BOX, 600)); axes[0].set_title("발화 전 3/14 (잔설)")
axes[1].imshow(thumb(post_img, vis_rgb, SNOW_BOX, 600)); axes[1].set_title("발화 후 4/8")
axes[2].imshow(thumb(severity, {"min": 1, "max": 6, "palette": [c[1:] for c in SEV_COLORS]}, SNOW_BOX, 600)); axes[2].set_title("dNBR 등급")
axes[3].imshow(thumb(snow.selfMask(), {"palette": ["00ffff"]}, SNOW_BOX, 600)); axes[3].set_title("눈 판정 (NDSI > 0.4)")
for a in axes: a.axis("off")
plt.tight_layout(); plt.savefig("07_snow_artifact.png", dpi=120, bbox_inches="tight"); plt.show()

# 잔설 상자를 30 m 격자로 받는다 (7-4에서도 이 파일을 쓴다)
SNOW_FILES = {n: f"{DATA_DIR}/snow_{n}.tif" for n in ["post", "dnbr", "ndsi", "firms"]}
ee_to_geotiff(post_img.select(BANDS), SNOW_BOX, SNOW_FILES["post"])
ee_to_geotiff(dnbr,                   SNOW_BOX, SNOW_FILES["dnbr"])
ee_to_geotiff(ndsi_pre,               SNOW_BOX, SNOW_FILES["ndsi"])
ee_to_geotiff(firms,                  SNOW_BOX, SNOW_FILES["firms"])

**코드 설명**
- `s_`로 시작하는 변수는 모두 잔설 상자의 것이다. `s_ok`는 쓸 수 있는 픽셀이다.
- dNBR > 0.27인 픽셀 면적을 세고, 눈 기준값 0.4·0.2로 눈 픽셀을 뺀 뒤 다시 센다.

In [ ]:
s_post = load_tif(SNOW_FILES["post"])[0]
s_dnbr = load_tif(SNOW_FILES["dnbr"])[0][0]
s_ndsi = load_tif(SNOW_FILES["ndsi"])[0][0]
s_fire = load_tif(SNOW_FILES["firms"])[0][0] > 0.5
s_ok = (s_post != NODATA).all(0) & (s_dnbr != NODATA) & np.isfinite(s_dnbr) & (s_post[3] > 0)

s_dnbr_burn = (s_dnbr > BURN_T) & s_ok
print(f"잔설 상자 '중간 이상 피해'(dNBR > {BURN_T}): {s_dnbr_burn.sum()*PIXEL_HA:,.0f} ha")
for t in [0.4, 0.2]:   # 눈 기준값을 바꿔 가며 비교한다
    print(f"눈 픽셀(NDSI > {t})을 뺀 뒤:  {(s_dnbr_burn & ~(s_ndsi > t)).sum()*PIXEL_HA:,.0f} ha")

> 기준 0.4는 확실한 눈만 잡아서 '피해'가 조금밖에 안 줄어든다. 3/14 진색 영상의 능선에 남은 얇은 눈을 놓치기 때문이다. 0.2로 낮추면 얇은 눈까지 잡아 더 줄지만, 눈이 아닌 것까지 가릴 위험도 커진다.
> 기준값 하나로 결과가 달라진다. **완벽한 마스크는 없다.** 발화 전 영상이 나쁘면 dNBR을 나중에 고치기가 어렵다.

### 7-4. 발화 전 영상이 나쁠 때: AI는 속을까?
잔설 상자는 **발화 전 영상이 나쁜 곳**이다. dNBR은 발화 전 영상을 쓰니 눈에 속는다. AI(MLP·U-Net)와 기준선은 발화 후 영상 한 장만 보니 눈을 본 적이 없다.
같은 상자에서 네 방법의 '피해'를 비교하고, FIRMS 화점으로 실제로 불이 났는지 확인한다.

**코드 설명**
- 7-3에서 받은 잔설 상자 파일(`s_post`, `s_dnbr`, `s_fire`)을 그대로 쓴다. 그래서 dNBR 면적이 7-3의 첫 줄과 같다.
- 여기서는 학습을 하지 않는다. 6장에서 학습을 마친 모델을 **처음 보는 곳**에 그대로 적용한다.
- 네 방법으로 '피해'를 판정한다. dNBR은 발화 전·후 두 장을, 기준선·MLP·U-Net은 발화 후 영상 한 장만 쓴다.
- MLP는 6-3의 `pixel_features()`로 표를 만들어 넣고, U-Net은 상자 전체를 32의 배수로 채워 한 번에 넣는다.
- 출력 읽기: 먼저 FIRMS 화점 면적을 본다. 화점이 없으면 이 상자의 '피해'는 가짜다. 그 가짜를 어느 방법이 얼마나 잡았는지 비교한다.

In [ ]:
s_thr_burn = (post_nbr(s_post) < NBR_T) & s_ok             # 한 장: 기준선
mlp.eval()
with torch.no_grad():                                       # 한 장: MLP
    s_mlp_prob = mlp(torch.from_numpy(pixel_features(s_post)).to(device)).cpu().numpy().reshape(s_ok.shape)
s_mlp_burn = (s_mlp_prob > 0.5) & s_ok
methods = [("dNBR (두 장)", s_dnbr_burn), ("기준선 NBR 문턱값 (한 장)", s_thr_burn), ("MLP (한 장)", s_mlp_burn)]
if "unet" in globals():                                     # 한 장: U-Net
    hs, ws = s_ok.shape
    xs = np.pad(unet_input(s_post, s_ok), ((0, 0), (0, (32 - hs % 32) % 32), (0, (32 - ws % 32) % 32)), mode="reflect")
    unet.eval()
    with torch.no_grad():
        s_unet_prob = torch.sigmoid(unet(torch.from_numpy(xs[None]).to(device)))[0, 0, :hs, :ws].cpu().numpy()
    methods.append(("U-Net (한 장)", (s_unet_prob > 0.5) & s_ok))

print(f"잔설 상자: FIRMS 화점 영역 {(s_fire & s_ok).sum()*PIXEL_HA:,.0f} ha")
for name, m in methods:
    print(f"  {name:24s} 피해 {m.sum()*PIXEL_HA:7,.0f} ha")

s_rgb = np.clip(np.stack([s_post[2], s_post[1], s_post[0]], -1) / 1600, 0, 1); s_rgb[~s_ok] = 1
fig, axes = plt.subplots(1, 1 + len(methods), figsize=(5 * (1 + len(methods)), 3.6))
axes[0].imshow(s_rgb); axes[0].set_title("발화 후 진색 4/8")
for ax, (name, m) in zip(axes[1:], methods):
    ax.imshow(np.where(s_ok, m, np.nan), cmap="RdYlGn_r", vmin=0, vmax=1); ax.set_title(name + " 피해")
for a in axes: a.axis("off")
plt.tight_layout(); plt.savefig("08_snow_ai.png", dpi=120, bbox_inches="tight"); plt.show()

> 이것이 이 실습에서 AI를 만든 이유다. **발화 전 영상이 쓸 만하지 않은 곳**에서는 두 장을 쓰는 dNBR이 틀린다. 발화 후 영상 한 장으로 배운 AI는 그 오류를 물려받지 않는다.
> MLP는 작은 점 몇 개를 잡지만, 주변을 보는 U-Net은 외딴 가짜 점도 거의 만들지 않는다. 7-1에서 본 두 모델의 차이가 여기서도 나타난다.
> 다만 AI도 완벽하지 않다. 학습 상자에서 dNBR이 틀린 곳까지 그대로 배우고, 학습한 지역·계절과 많이 다른 곳에서는 틀릴 수 있다. 그래서 FIRMS 같은 독립 자료로 확인한다.

---
## 섹션 8. 요약

**코드 설명**
- 지금까지 계산한 결과를 한 번에 출력한다. 기준선·MLP·U-Net의 검증 정확도와, 잔설 상자에서 dNBR과 MLP가 잡은 '피해'를 함께 본다.
- 피해지 NDVI: dNBR 피해 픽셀들의 발화 전·후 NDVI 평균과 변화율이다.

In [ ]:
print("=" * 62)
print("  2025 의성 산불 · Sentinel-2 + AI 분석 요약 (AI 검증 영역)")
print("=" * 62)
print(f"  영상: 발화 전 {PRE_START} / 발화 후 {POST_START}  |  격자 30 m, {H}×{W}")
print(f"  분석 면적: {valid.sum()*PIXEL_HA:,.0f} ha")
print(f"  dNBR 피해(>0.27): {dnbr_burn.sum()*PIXEL_HA:,.0f} ha")
print(f"  AI 입력: 발화 후 영상 한 장 | 학습: 상자 {len(TRAIN_BOXES)}개 / 검증: 검증 영역 (dNBR 정답 기준)")
print(f"  기준선(발화 후 NBR < {NBR_T:.2f}): {thr_burn.sum()*PIXEL_HA:,.0f} ha  | 검증 정확도 {thr_val_acc:.1f}%")
print(f"  MLP 피해(>0.5):   {mlp_burn.sum()*PIXEL_HA:,.0f} ha  | 검증 정확도 {hist['val_acc'][-1]:.1f}%")
if "unet_burn" in globals():
    print(f"  U-Net 피해(>0.5): {unet_burn.sum()*PIXEL_HA:,.0f} ha  | 검증 정확도 {unet_val_acc:.1f}%")
if "s_mlp_burn" in globals():
    print(f"  잔설 상자 (화점 {(s_fire & s_ok).sum()*PIXEL_HA:,.0f} ha): dNBR {s_dnbr_burn.sum()*PIXEL_HA:,.0f} ha / MLP {s_mlp_burn.sum()*PIXEL_HA:,.0f} ha")
nd_b, nd_a = pre_idx[0][dnbr_burn].mean(), post_idx[0][dnbr_burn].mean()
print(f"  피해지 NDVI: {nd_b:.3f} → {nd_a:.3f} ({(nd_a-nd_b)/abs(nd_b)*100:+.0f}%)")
print("=" * 62)

## 심화 과제
1. `BURN_T`를 0.27 → 0.44로 올리면 MLP 결과가 어떻게 달라지나?
2. `TRAIN_BOXES`를 **A 하나만** 남기면 검증 영역 정확도가 어떻게 달라지나? 피해가 거의 없는 **C·D만** 쓰면?
3. 발화 전 기간을 `2025-03-01`~`2025-03-21`로 넓혀 중앙값 합성을 하면 잔설 효과가 줄어드나?
4. 산림청 **임상도**(산림공간정보서비스)를 겹쳐 침엽수림과 활엽수림의 피해 심각도를 비교해 보자.
5. Sentinel-5P 일산화탄소(`COPERNICUS/S5P/OFFL/L3_CO`)로 3월 22~30일 연기의 확산을 지도로 그려 보자.
6. **회복 모니터링**: 2024-01 ~ 2026-09 매달 맑은 영상의 NDVI를 탄 숲(dNBR>0.44)과 안 탄 숲으로 나눠 그려 보자.
   (힌트: `ee.ImageCollection(...).filterDate(...)`를 달마다 돌리고 `reduceRegion(ee.Reducer.median())`).
   강의 자료의 '숲은 돌아오고 있나' 그래프와 같은 모양이 나오는지 비교한다.
7. **가시광만으로 학습하면?** MLP 입력을 B2·B3·B4 세 개로 줄여 다시 학습해 보자. 검증 정확도가 얼마나 떨어지나? 근적외·단파적외가 왜 필요한지 알 수 있다.
   (힌트: `X_tr[:, :3]`, `X_va[:, :3]`, `all_feats[:, :3]`을 쓰고 `BurnAreaMLP(in_features=3)`으로 만든다.)

## 참고 자료
- UN-SPIDER Burn Severity 권장 실무: https://un-spider.org/advisory-support/recommended-practices/burn-severity
- Sentinel-2 사용자 가이드: https://sentinels.copernicus.eu/web/sentinel/user-guides/sentinel-2-msi
- Earth Engine 데이터 카탈로그 (S2_SR_HARMONIZED, FIRMS, S5P): https://developers.google.com/earth-engine/datasets
- 김준우·김덕진(2020) 위성영상-AI 기반 재난모니터링과 실현 가능한 준실시간 통합 재난모니터링 시스템, 한국지리정보학회지 23(3)
- 채한성·최진무(2024) Sentinel-2 위성영상과 U-Net을 이용한 산불 피해지 추출 방법 연구, 대한지리학회지 59(2)

> 이 실습은 무료 공개 자료(Copernicus Sentinel-2, NASA FIRMS)와 무료 플랫폼(Google Earth Engine, Colab)만 쓴다. 같은 방법을 다른 산불에도 적용할 수 있다.